In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("cleaned_data.csv")
print("Data loaded:", df.shape)

# Features and target
X = df.drop("Patv", axis=1)
y = df["Patv"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale (for Linear Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Everything ready ✅")

Data loaded: (3373786, 11)
Everything ready ✅


In [2]:
# Use 20% of data to make training faster
df = df.sample(frac=0.2, random_state=42)
print("Using samples:", df.shape)

Using samples: (674757, 11)


In [ ]:
X = df.drop("Patv", axis=1)   # All columns except power output
y = df["Patv"]                 #we want to predict

print("Features (X):", X.columns.tolist())
print("Target (y): Patv — Power Output")

Features (X): ['Wspd', 'Wdir', 'Etmp', 'Itmp', 'Ndir', 'Pab1', 'Pab2', 'Pab3', 'Prtv', 'yaw_error']
Target (y): Patv — Power Output


In [4]:
# 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing  samples: {len(X_test)}")

Training samples: 539805
Testing  samples: 134952


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# 2. Decision Tree
dt = DecisionTreeRegressor(max_depth=10, random_state=42)
dt.fit(X_train, y_train)

# 3. Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print("All models trained ✅")

In [ ]:
def evaluate(name, model, X_t, y_t):
    pred = model.predict(X_t)
    mae  = mean_absolute_error(y_t, pred)
    rmse = np.sqrt(mean_squared_error(y_t, pred))
    r2   = r2_score(y_t, pred)
    print(f"{name:20s} | MAE: {mae:7.2f} | RMSE: {rmse:7.2f} | R²: {r2:.4f}")
    return pred

print(f"{'Model':20s} | {'MAE':>10} | {'RMSE':>10} | {'R²':>8}")
print("-" * 55)
pred_lr = evaluate("Linear Regression",   lr, X_test_scaled, y_test)
pred_dt = evaluate("Decision Tree",       dt, X_test,        y_test)
pred_rf = evaluate("Random Forest",       rf, X_test,        y_test)

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_test, pred_rf, alpha=0.2, s=5, color='steelblue')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel("Actual Power (kW)")
plt.ylabel("Predicted Power (kW)")
plt.title("Random Forest: Actual vs Predicted")
plt.legend()
plt.savefig("03_actual_vs_predicted.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(rf, "models/wind_model.pkl")
print("Model saved ✅")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import joblib
import os

# Load data
df = pd.read_csv("cleaned_data.csv")
X = df.drop("Patv", axis=1)
y = df["Patv"]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train
print("Training model... (this may take 2-3 minutes)")
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Save
os.makedirs("models", exist_ok=True)
joblib.dump(rf, "models/wind_model.pkl")
print("Model saved ✅ — check your models/ folder!")

In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)
joblib.dump(rf, "models/wind_model.pkl")
print("Model saved ✅")